# 07 — Ministry of Agriculture Prices | Gold KPIs (AGR)

| Table | Grain | KPIs |
|---|---|---|
| `gold.agri_price_monthly` | channel × market × product × unit × month | **AGR-001** average price, **AGR-002** MoM %, **AGR-003** YoY %, **AGR-004** YTD %, **AGR-005** 12-month volatility, **AGR-007** market price difference % |
| `gold.agri_rubro_monthly` | channel × rubro (product group) × month | products quoted, average and median MoM / YoY of the products in the group, share of products up and down |

| Decision | Rule |
|---|---|
| Monthly price | Average of the weekly quotes of the month for each market (`weeks_quoted` kept). |
| All retail markets | Extra series `market = 'Promedio mercados'`: equal-weight average of the market monthly prices (each market counts once, whatever its number of weeks). |
| Comparability | A series is channel + market + product + unit; prices in different units are never compared. MoM needs the previous calendar month, YoY the same month a year earlier, YTD the previous December, in the same series; otherwise NULL. |
| AGR-005 volatility | Standard deviation of the MoM % over the last 12 months, with at least 6 MoM values; otherwise NULL. |
| AGR-007 | **Market** price difference: `(market price / Promedio mercados − 1) × 100` for each retail market. The source has Gran Santo Domingo markets, not regions, so the catalog's *regional* difference is delivered at market level and labelled as such. |
| Rubro | Each product gets its most frequent rubro, so a product never moves between groups because of a source typo. |
| AGR-006 | Seasonal index not calculated: methodology not defined in the catalog. |

**Status:** Development.

In [0]:
import uuid, json
from datetime import datetime, timezone
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG = "rd_economic_intelligence"
SOURCE = f"{CATALOG}.silver.ma_agri_prices_weekly"
T_MONTH = f"{CATALOG}.gold.agri_price_monthly"
T_RUBRO = f"{CATALOG}.gold.agri_rubro_monthly"
RESULTS = f"{CATALOG}.quality.dq_rule_results"
RUN_ID, RUN_TS = str(uuid.uuid4()), datetime.now(timezone.utc)
ALL = "Promedio mercados"

s = spark.table(SOURCE)
mode_rubro = (s.groupBy("channel", "product", "rubro").count()
              .withColumn("_r", F.row_number().over(Window.partitionBy("channel", "product").orderBy(F.desc("count"), "rubro")))
              .filter("_r = 1").select("channel", "product", "rubro"))

market_m = (s.groupBy("channel", "market", "product", "unit", "reference_month")
            .agg(F.avg("price_dop").alias("avg_price"), F.countDistinct("week_of_month").alias("weeks_quoted")))
all_m = (market_m.filter("channel = 'retail'").groupBy("channel", "product", "unit", "reference_month")
         .agg(F.avg("avg_price").alias("avg_price"), F.count("*").alias("markets_quoted"), F.sum("weeks_quoted").alias("weeks_quoted"))
         .withColumn("market", F.lit(ALL)))
monthly = (market_m.withColumn("markets_quoted", F.lit(1)).unionByName(all_m)
           .join(mode_rubro, ["channel", "product"]))

series = ["channel", "market", "product", "unit"]
w = Window.partitionBy(*series).orderBy("reference_month")
def prior(months):
    p = monthly.select(*series, F.add_months("reference_month", months).alias("reference_month"), F.col("avg_price").alias(f"p{months}"))
    return p
dec = monthly.filter(F.month("reference_month") == 12).select(*series, F.year("reference_month").alias("_y"), F.col("avg_price").alias("p_dec"))
g = (monthly.join(prior(1), series + ["reference_month"], "left")
            .join(prior(12), series + ["reference_month"], "left")
            .withColumn("_y", F.year("reference_month") - 1).join(dec, series + ["_y"], "left").drop("_y")
            .withColumn("mom_pct", (F.col("avg_price") / F.col("p1") - 1) * 100)
            .withColumn("yoy_pct", (F.col("avg_price") / F.col("p12") - 1) * 100)
            .withColumn("ytd_pct", F.when(F.month("reference_month") != 12, (F.col("avg_price") / F.col("p_dec") - 1) * 100)
                                    .otherwise((F.col("avg_price") / F.col("p12") - 1) * 100)))
# AGR-005: stddev of MoM over the last 12 months (by calendar month, at least 6 values)
w12 = Window.partitionBy(*series).orderBy(F.col("_mi")).rangeBetween(-11, 0)
g = (g.withColumn("_mi", F.year("reference_month") * 12 + F.month("reference_month"))
      .withColumn("_n12", F.count("mom_pct").over(w12)).withColumn("_sd12", F.stddev_samp("mom_pct").over(w12))
      .withColumn("volatility_12m_pct", F.when(F.col("_n12") >= 6, F.col("_sd12"))))
# AGR-007: retail market vs average of markets, same product, unit and month
ref = g.filter(F.col("market") == ALL).select("channel", "product", "unit", "reference_month", F.col("avg_price").alias("p_all"))
g = (g.join(ref, ["channel", "product", "unit", "reference_month"], "left")
      .withColumn("market_diff_pct", F.when((F.col("channel") == "retail") & (F.col("market") != ALL), (F.col("avg_price") / F.col("p_all") - 1) * 100)))
dec4 = lambda c: F.round(c, 4).cast("decimal(12,4)")
gold = g.select("channel", "market", "rubro", "product", "unit", "reference_month",
                F.round("avg_price", 2).cast("decimal(12,2)").alias("avg_price_dop"), "weeks_quoted", "markets_quoted",
                dec4(F.col("mom_pct")).alias("mom_pct"), dec4(F.col("yoy_pct")).alias("yoy_pct"), dec4(F.col("ytd_pct")).alias("ytd_pct"),
                dec4(F.col("volatility_12m_pct")).alias("volatility_12m_pct"), dec4(F.col("market_diff_pct")).alias("market_diff_pct"),
                F.lit(RUN_ID).alias("_gold_run_id"), F.lit(RUN_TS).cast("timestamp").alias("_gold_processed_ts"))

# Group view: retail uses the all-markets series, wholesale its single series
base = gold.filter(((F.col("channel") == "retail") & (F.col("market") == ALL)) | (F.col("channel") == "wholesale"))
rubro = (base.groupBy("channel", "rubro", "reference_month").agg(
            F.countDistinct(F.concat_ws("|", "product", "unit")).alias("products_quoted"),
            F.count("mom_pct").alias("products_with_mom"),
            dec4(F.avg("mom_pct")).alias("avg_mom_pct"), dec4(F.percentile_approx("mom_pct", 0.5)).alias("median_mom_pct"),
            dec4(F.avg("yoy_pct")).alias("avg_yoy_pct"), dec4(F.percentile_approx("yoy_pct", 0.5)).alias("median_yoy_pct"),
            dec4(F.avg(F.when(F.col("mom_pct") > 0, 100.0).when(F.col("mom_pct").isNotNull(), 0.0))).alias("share_up_pct"),
            dec4(F.avg(F.when(F.col("mom_pct") < 0, 100.0).when(F.col("mom_pct").isNotNull(), 0.0))).alias("share_down_pct"))
         .withColumn("_gold_run_id", F.lit(RUN_ID)).withColumn("_gold_processed_ts", F.lit(RUN_TS).cast("timestamp")))

In [0]:
# Validation before writing
checks = []
def add(table, name, rtype, tested, failed):
    checks.append((RUN_ID, table, name, rtype, int(tested), int(failed), "PASS" if failed == 0 else ("FAIL" if rtype == "enforced" else "WARN"), RUN_TS))
n = gold.count()
expected = s.select("channel", "market", "product", "unit", "reference_month").distinct().count() + all_m.count()
add(T_MONTH, "row_count_matches_silver_series", "enforced", n, abs(n - expected))
add(T_MONTH, "unique_series_month", "enforced", n, n - gold.select(*series, "reference_month").distinct().count())
add(T_MONTH, "price_positive", "enforced", n, gold.filter("avg_price_dop <= 0").count())
add(T_MONTH, "market_diff_only_retail_markets", "enforced", n, gold.filter((F.col("market_diff_pct").isNotNull()) & ((F.col("channel") != "retail") | (F.col("market") == ALL))).count())
add(T_MONTH, "abs_mom_above_100pct", "observational", gold.filter("mom_pct IS NOT NULL").count(), gold.filter("abs(mom_pct) > 100").count())
nr = rubro.count()
add(T_RUBRO, "unique_channel_rubro_month", "enforced", nr, nr - rubro.select("channel", "rubro", "reference_month").distinct().count())
for c in checks: print(f"  {c[6]:<4} {c[1].split('.')[-1]:<20} {c[2]:<34} failed={c[5]:,}/{c[4]:,}")
assert not [c for c in checks if c[6] == "FAIL"], "Enforced Gold rules failed; nothing written"

gold.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(T_MONTH)
rubro.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(T_RUBRO)
spark.sql(f"COMMENT ON TABLE {T_MONTH} IS 'Gold: Ministry of Agriculture monthly prices by channel, market, product and unit with AGR-001..005 and AGR-007 (market difference vs average of Gran Santo Domingo markets).'")
spark.sql(f"COMMENT ON TABLE {T_RUBRO} IS 'Gold: Ministry of Agriculture price changes by product group (rubro) and month: products quoted, average/median MoM and YoY, share of products up/down.'")
spark.createDataFrame(checks, "run_id string, target_table string, rule_name string, rule_type string, records_tested bigint, records_failed bigint, status string, run_ts timestamp").write.mode("append").saveAsTable(RESULTS)

latest = spark.sql(f"SELECT max(reference_month) m FROM {T_RUBRO}").first()["m"]
snap = spark.sql(f"SELECT channel, rubro, products_quoted, avg_mom_pct, avg_yoy_pct FROM {T_RUBRO} WHERE reference_month = '{latest}' ORDER BY channel, products_quoted DESC").collect()
dbutils.notebook.exit(json.dumps({"status": "SUCCESS", "monthly_rows": n, "rubro_rows": nr, "latest_month": str(latest),
    "checks": {c[2]: f"{c[6]} ({c[5]}/{c[4]})" for c in checks}, "latest_by_rubro": [r.asDict() for r in snap]}, default=str, ensure_ascii=False))